In [ ]:
import pandas as pd

data = pd.read_csv("data/songs_train.csv")
test_data = pd.read_csv("data/songs_test.csv")

#make pandas show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.duplicated().sum()

In [ ]:
data.nunique()

In [ ]:
data.isnull().sum()

In [ ]:
data.dropna(inplace=True)
data.shape

In [ ]:
data['track_album_release_date'] = pd.to_datetime(data['track_album_release_date'], format='mixed' )
test_data['track_album_release_date'] = pd.to_datetime(test_data['track_album_release_date'], format='mixed')

In [ ]:
min_date = data['track_album_release_date'].min()
max_date = data['track_album_release_date'].max()
data['track_album_release_date_normalized'] = (data['track_album_release_date'] - min_date) / (max_date - min_date)
data.drop('track_album_release_date', axis=1, inplace=True)

test_data['track_album_release_date_normalized'] = (test_data['track_album_release_date'] - min_date) / (max_date - min_date)
test_data.drop('track_album_release_date', axis=1, inplace=True)

drop some features because data is very big


In [ ]:
data.drop(['id','track_id', 'track_name', 'track_album_name', 'track_album_id', 'track_artist', 'playlist_name', 'playlist_id'], axis=1, inplace=True)
test_data.drop(['id','track_id', 'track_name', 'track_album_name', 'track_album_id', 'track_artist', 'playlist_name', 'playlist_id'], axis=1, inplace=True)

In [ ]:
test_data.shape

In [ ]:
# strip and lower string columns to ensure uniformity before one hot encoding
from sklearn.preprocessing import OneHotEncoder

def clean_string_columns_encode(data, test_data):
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = data[col].str.strip().str.lower()
        test_data[col] = test_data[col].str.strip().str.lower()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_data = encoder.fit_transform(data.select_dtypes(include=['object']))
    encoded_test_data = encoder.transform(test_data.select_dtypes(include=['object']))
    encoded_cols = encoder.get_feature_names_out(data.select_dtypes(include=['object']).columns)
    encoded_df = pd.DataFrame(encoded_data, columns=encoded_cols, index=data.index)
    encoded_test_df = pd.DataFrame(encoded_test_data, columns=encoded_cols, index=test_data.index)
    data = pd.concat([data.drop(columns=data.select_dtypes(include=['object']).columns), encoded_df], axis=1)
    test_data = pd.concat([test_data.drop(columns=test_data.select_dtypes(include=['object']).columns), encoded_test_df], axis=1)
    
    return data, test_data

data, test_data = clean_string_columns_encode(data, test_data)

data.shape

In [ ]:
test_data.shape

In [ ]:
num_classes = data['track_popularity'].nunique()

In [ ]:
from sklearn.model_selection import train_test_split
X = data.drop('track_popularity', axis=1)
y = data['track_popularity']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(test_data.values)


Neural Network model with py torch

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

batch_size = 256

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
val_dataset = torch.utils.data.TensorDataset(X_val_tensor, y_val_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


X_test = torch.tensor(X_test, dtype=torch.float32)
test_dataset = torch.utils.data.TensorDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(X_train.shape[1], 128),
            # nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            # nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0,2),

            nn.Linear(64, 32),
            # nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        
        )

    def forward(self, x):
        x = x.to(device)
        return self.layers(x)
        
model = NeuralNetwork().to(device)
print(model)
        

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.006)

In [ ]:
from traitlets import This


def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            # Print RMSE (sqrt of MSE) as it's more interpretable (on the same scale as popularity)
            print(f"RMSE loss: {loss ** 0.5:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    test_rmse = torch.sqrt(torch.tensor(test_loss)) # Convert to RMSE
    print(f"Validation Error: \n Avg RMSE: {test_rmse:>8f} \n")
    # return test_loss # Return loss for early stopping

In [ ]:
epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
    test(val_loader, model, loss_fn)
print("Done!")

In [ ]:
print(val_loader)

In [ ]:
import torch.nn.functional as F

mse_loss = nn.MSELoss()
total_mse = 0
num_batches = len(val_loader)

model.eval()
with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)

        pred = model(X)
        # One-hot encode y to match pred’s second dimension (101)

        batch_mse = mse_loss(pred, y)
        total_mse += batch_mse.item()

avg_mse = total_mse / num_batches
print(f"Average MSE on validation set: {avg_mse:.6f}")


In [ ]:
test_id = pd.read_csv("data/songs_test.csv")['id']

test_id.head()

In [ ]:
# (Assuming 'test_id' is defined earlier, e.g., test_id = test_data['id'] 
# *before* you dropped it. Make sure to save it first!)

# You need to save test IDs before dropping them!
# Do this right after loading the test data.
# test_data = pd.read_csv(...)
# test_id = test_data['id'] 

predictions = []
model.eval()
with torch.no_grad():
    for X in test_loader:
        X = X[0].to(device) # X[0] because test_loader only has X, not (X, y)
        
        pred = model(X)
        
        # Squeeze to remove the [batch_size, 1] dim and make it [batch_size]
        # Round to the nearest integer
        # Clamp values to be between 0 and 100
        # Move to CPU and numpy
        predicted_values = torch.clamp(torch.round(pred.squeeze()), 0, 100).cpu().numpy()
        
        predictions.extend(predicted_values)

# Create submission
submission_df = pd.DataFrame({'id': test_id, 'track_popularity': predictions})
# Ensure predictions are integers for submission
submission_df['track_popularity'] = submission_df['track_popularity'].astype(int) 
submission_df.to_csv('submission.csv', index=False)

print("Submission file created!")